### Middleware

Middleware provides a way to more tightly control what happens inside the agent. Middleware is useful for the following:
- Tracking agent behavior with logging, analytics, and debugging.
- Transforming prompts, tool selection, and output formatting.
- Adding retries, fallbacks, and early termination logic.
- Applying rate limits, guardrails, and PII detection.

In [38]:
import os
from dotenv import load_dotenv
load_dotenv()

os.environ["GROQ_API_KEY"] = os.getenv("GROQ_API_KEY")

### Summarization MiddleWare
Automatically summarize conversation history when approaching token limits, preserving recent messages while compressing older context. Summarization is useful for the following:
- Long-running conversations that exceed context windows.
- Multi-turn dialogues with extensive history.
- Applications where preserving full conversation context matters.

In [39]:
from langchain.agents import create_agent
from langchain.agents.middleware import SummarizationMiddleware
from langgraph.checkpoint.memory import InMemorySaver
from langchain_core.messages import HumanMessage, SystemMessage

### Messagebased summarization
agent=create_agent(
    model="groq:qwen/qwen3-32b",
    checkpointer=InMemorySaver(),
    # Config of model
    middleware=[
        SummarizationMiddleware(
            model="groq:qwen/qwen3-32b",    # Generally cheaper model
            trigger=("messages",10),        # when should the summarization happen?
            keep=("messages",4)             # How many top messages to keep?
        )
    ]
)

# ?

In [40]:
### Run with thread id
# Creating thread
config={"configurable":{"thread_id":"test-1"}}

In [41]:
# Alternative test data
questions = [
    "What is 2+2?",
    "What is 10*5?",
    "What is 100/4?",
    "What is 15-7?",
    "What is 3*3?",
    "What is 4*4?",
]

for q in questions:
    response=agent.invoke({"messages":[HumanMessage(content=q)]},config)
    print(f"Messages: {response}")
    print(f"Messages: {len(response['messages'])}")     # To see that when message length becomes 10 -> summarization occurs


Messages: {'messages': [HumanMessage(content='What is 2+2?', additional_kwargs={}, response_metadata={}, id='f785454a-5f8d-4848-9f3d-5cefdd289316'), AIMessage(content='<think>\nOkay, so I need to figure out what 2+2 is. Let me start by recalling basic addition. When you add two numbers together, you\'re combining their quantities. So 2 plus 2 should be... 4? Wait, is there a trick here? Sometimes people ask this question to check for basic math skills or maybe to see if someone is paying attention. Let me think again.\n\nStarting with the number 2 and adding another 2. If I count two more from 2, that would be 2, 3, 4. Yeah, that seems right. Maybe I can use objects to visualize. If I have two apples and someone gives me two more apples, how many do I have in total? That\'s four apples. So 2+2 must equal 4. \n\nHmm, could it be something else? In some contexts, like different number bases, the result might change. For example, in base 3, 2+2 would be 11. But the question doesn\'t speci

## token Size based summarization

# CRAZY

In [ ]:
from langchain.agents import create_agent
from langchain.agents.middleware import SummarizationMiddleware
from langchain_core.tools import tool
from langchain_core.messages import HumanMessage
from langgraph.checkpoint.memory import InMemorySaver

@tool
def search_hotels(city: str) -> str:
    """Search hotels."""
    return f"Hotels in {city}: Grand Hotel $350, City Inn $180, Budget Stay $75"

# LOW fraction for testing!
agent = create_agent(
    model="groq:qwen/qwen3-32b",
    tools=[search_hotels],
    checkpointer=InMemorySaver(),
    middleware=[
        SummarizationMiddleware(
            model="groq:qwen/qwen3-32b",
            trigger=("token", 550),  # 0.5% = ~640 tokens
            keep=("token", 200),     # 0.2% = ~256 tokens
        ),
    ],
)

config = {"configurable": {"thread_id": "test-1"}}

# Token counter
def count_tokens(messages):
    return sum(len(str(m.content)) for m in messages) // 4

Paris: ~70 tokens (0.0547%), 4 msgs
[HumanMessage(content='Hotels in Paris', additional_kwargs={}, response_metadata={}, id='ca74cc45-e9ca-4b68-84b0-c79aa546a539'), AIMessage(content='', additional_kwargs={'reasoning_content': "Okay, the user is asking for hotels in Paris. Let me check the available tools. There's a function called search_hotels that requires a city parameter. The user specified Paris, so I need to call that function with the city set to Paris. I should make sure the arguments are correctly formatted as JSON within the tool_call tags. Let me structure the response properly.\n", 'tool_calls': [{'id': '49wmwdvwj', 'function': {'arguments': '{"city":"Paris"}', 'name': 'search_hotels'}, 'type': 'function'}]}, response_metadata={'token_usage': {'completion_tokens': 100, 'prompt_tokens': 147, 'total_tokens': 247, 'completion_time': 0.142108596, 'completion_tokens_details': {'reasoning_tokens': 75}, 'prompt_time': 0.007106297, 'prompt_tokens_details': None, 'queue_time': 0.05

In [42]:
# Run test
cities = ["Paris", "London", "Tokyo", "New York", "Dubai", "Singapore"]

for city in cities:
    response = agent.invoke(
        {"messages": [HumanMessage(content=f"Find hotels in {city}")]},
        config=config
    )
    
    tokens = count_tokens(response["messages"])
    print(f"{city}: ~{tokens} tokens, {len(response['messages'])} messages")
    print(f"{(response['messages'])}")

Paris: ~1930 tokens, 8 messages
[HumanMessage(content='Here is a summary of the conversation to date:\n\n<think>\nOkay, let\'s see. The user is asking "What is 15-7?" I need to figure out the answer. Let me start by recalling basic subtraction. 15 minus 7... Hmm. If I count down from 15 seven times, that should give me the result. So 15 minus 7 is the same as starting at 15 and subtracting 7. Let me break it down step by step. 15 minus 5 is 10, then minus 2 more is 8. So 15 minus 7 equals 8. Alternatively, I can think of it as adding 8 to 7 to get 15, which confirms that 15 minus 7 is indeed 8. I don\'t think there\'s any trick here unless there\'s a specific context, but the question seems straightforward. The user might just want a quick answer, so I\'ll state it clearly.\n</think>\n\nThe result of subtracting 7 from 15 is **8**. \n\nIn standard arithmetic, 15 minus 7 equals 8, as 7 added to 8 gives 15 (7 + 8 = 15).\n\n**Answer:** 8', additional_kwargs={'lc_source': 'summarization'},

### Fraction based summarization

In [43]:
from langchain.agents import create_agent
from langchain.agents.middleware import SummarizationMiddleware
from langchain_core.tools import tool
from langchain_core.messages import HumanMessage
from langgraph.checkpoint.memory import InMemorySaver

@tool
def search_hotels(city: str) -> str:
    """Search hotels."""
    return f"Hotels in {city}: Grand Hotel $350, City Inn $180, Budget Stay $75"

# LOW fraction for testing!
agent = create_agent(
    model="groq:qwen/qwen3-32b",
    tools=[search_hotels],
    checkpointer=InMemorySaver(),
    middleware=[
        SummarizationMiddleware(
            model="groq:qwen/qwen3-32b",
            trigger=("fraction", 0.005),  # 0.5% = ~640 tokens
            keep=("fraction", 0.002),     # 0.2% = ~256 tokens
        ),
    ],
)

config = {"configurable": {"thread_id": "test-1"}}

# Token counter
def count_tokens(messages):
    return sum(len(str(m.content)) for m in messages) // 4

In [44]:
# Test
cities = ["Paris", "London", "Tokyo", "New York", "Dubai", "Singapore"]

for city in cities:
    response = agent.invoke(
        {"messages": [HumanMessage(content=f"Hotels in {city}")]},
        config=config
    )
    tokens = count_tokens(response["messages"])
    fraction = tokens / 128000  # groq:qwen/qwen3-32b context
    print(f"{city}: ~{tokens} tokens ({fraction:.4%}), {len(response['messages'])} msgs")
    print(response['messages'])

Paris: ~69 tokens (0.0539%), 4 msgs
[HumanMessage(content='Hotels in Paris', additional_kwargs={}, response_metadata={}, id='2675da7b-3956-485e-a7d3-b79622ca6f4e'), AIMessage(content='', additional_kwargs={'reasoning_content': "Okay, the user is asking for hotels in Paris. Let me check the available tools. There's a function called search_hotels that requires a city parameter. Since the user specified Paris, I need to call that function with the city set to Paris. I'll make sure the arguments are correctly formatted as JSON within the tool_call tags.\n", 'tool_calls': [{'id': 'p7aan3816', 'function': {'arguments': '{"city":"Paris"}', 'name': 'search_hotels'}, 'type': 'function'}]}, response_metadata={'token_usage': {'completion_tokens': 93, 'prompt_tokens': 147, 'total_tokens': 240, 'completion_time': 0.137187299, 'completion_tokens_details': {'reasoning_tokens': 68}, 'prompt_time': 0.014613103, 'prompt_tokens_details': None, 'queue_time': 0.391002757, 'total_time': 0.151800402}, 'mode

### Human In the Loop MiddleWare

Pause agent execution for human approval, editing, or rejection of tool calls before they execute. Human-in-the-loop is useful for the following:
- High-stakes operations requiring human approval (e.g. database writes, financial transactions).
- Compliance workflows where human oversight is mandatory.
- Long-running conversations where human feedback guides the agent.

# CRAZY

In [14]:
from langchain.agents import create_agent
from langchain.agents.middleware import HumanInTheLoopMiddleware
from langgraph.checkpoint.memory import InMemorySaver

def read_email_tool(email_id: str) -> str:
    """Mock function to read an email by its ID."""
    return f"Email content for ID: {email_id}"

def send_email_tool(recipient: str, subject: str, body: str) -> str:
    """Mock function to send an email."""
    return f"Email sent to {recipient} with subject '{subject}'"

In [ ]:
agent=create_agent(
    model="groq:qwen/qwen3-32b",
    tools=[read_email_tool,send_email_tool],
    checkpointer=InMemorySaver(),
    middleware=[
        HumanInTheLoopMiddleware(
            interrupt_on={
                "send_email_tool":{
                    # Choice for Human
                    "allowed_decisions":["approve","edit","reject"]
                },
                "read_email_tool":False,            # Dont need interrupt to read
            }
        )
    ]
)

In [16]:
config = {"configurable": {"thread_id": "test-approve"}}
# Step 1: Request
result = agent.invoke(
    {"messages": [HumanMessage(content="Send email to john@test.com with subject 'Hello' and body 'How are you?'")]},
    config=config
)

In [22]:
result

{'messages': [HumanMessage(content="Send email to john@test.com with subject 'Hello' and body 'How are you?'", additional_kwargs={}, response_metadata={}, id='1e7366ae-1d85-41d2-b4d8-a09e8e1a0b7e'),
  AIMessage(content='', additional_kwargs={'reasoning_content': "Okay, the user wants to send an email to john@test.com with the subject 'Hello' and body 'How are you?'. Let me check the available tools. There's the send_email_tool which requires recipient, subject, and body. All three are required. The parameters are all strings. The user provided all three pieces of information: recipient is john@test.com, subject is Hello, body is How are you? So I need to call send_email_tool with those arguments. No issues here. Just format the JSON correctly.\n", 'tool_calls': [{'id': 'p81m2n564', 'function': {'arguments': '{"body":"How are you?","recipient":"john@test.com","subject":"Hello"}', 'name': 'send_email_tool'}, 'type': 'function'}]}, response_metadata={'token_usage': {'completion_tokens': 1

In [23]:
from langgraph.types import Command
# Step 2: Approve
if "__interrupt__" in result:
    print("⏸️ Paused! Approving...")
    
    result = agent.invoke(
        Command(
            resume={
                "decisions": [
                    {"type": "approve"}
                ]
            }
        ),
        config=config
    )
    
    print(f"✅ Result: {result['messages'][-1].content}")

⏸️ Paused! Approving...
✅ Result: Your email has been sent to john@test.com with the subject "Hello". Let me know if you need anything else! 😊


In [24]:
result

{'messages': [HumanMessage(content="Send email to john@test.com with subject 'Hello' and body 'How are you?'", additional_kwargs={}, response_metadata={}, id='1e7366ae-1d85-41d2-b4d8-a09e8e1a0b7e'),
  AIMessage(content='', additional_kwargs={'reasoning_content': "Okay, the user wants to send an email to john@test.com with the subject 'Hello' and body 'How are you?'. Let me check the available tools. There's the send_email_tool which requires recipient, subject, and body. All three are required. The parameters are all strings. The user provided all three pieces of information: recipient is john@test.com, subject is Hello, body is How are you? So I need to call send_email_tool with those arguments. No issues here. Just format the JSON correctly.\n", 'tool_calls': [{'id': 'p81m2n564', 'function': {'arguments': '{"body":"How are you?","recipient":"john@test.com","subject":"Hello"}', 'name': 'send_email_tool'}, 'type': 'function'}]}, response_metadata={'token_usage': {'completion_tokens': 1

### Reject

In [29]:
from langchain.agents import create_agent
from langchain.agents.middleware import HumanInTheLoopMiddleware
from langgraph.checkpoint.memory import InMemorySaver


def read_email_tool(email_id: str) -> str:
    """Mock function to read an email by its ID."""
    return f"Email content for ID: {email_id}"

def send_email_tool(recipient: str, subject: str, body: str) -> str:
    """Mock function to send an email."""
    return f"Email sent to {recipient} with subject '{subject}'"

agent = create_agent(
    model="groq:qwen/qwen3-32b",
    tools=[read_email_tool,send_email_tool],
    checkpointer=InMemorySaver(),
    middleware=[
        HumanInTheLoopMiddleware(
            interrupt_on={
                "send_email_tool": {
                    "allowed_decisions": ["approve", "edit", "reject"],
                },
                "read_email_tool": False,
            }
        ),
    ],
)

# CRAZY

In [30]:
config = {"configurable": {"thread_id": "test-reject"}}
# Step 1: Request
result = agent.invoke(
    {"messages": [HumanMessage(content="Send email to john@test.com with subject 'Hello' and body 'How are you?'")]},
    config=config)

In [31]:
# Step 2: Reject
if "__interrupt__" in result:
    print("⏸️ Paused! Approving...")
    
    result = agent.invoke(
        Command(
            resume={
                "decisions": [
                    {"type": "reject"}
                ]
            }
        ),
        config=config
    )
    
    print(f"✅ Result: {result['messages'][-1].content}")

⏸️ Paused! Approving...
✅ Result: The email to john@test.com could not be sent (tool call rejected). Would you like me to:
1. Attempt to resend with adjusted parameters?
2. Check if there's an existing email draft using the read_email_tool?
3. Help troubleshoot the issue? 

Let me know how you'd like to proceed.


In [32]:
result

{'messages': [HumanMessage(content="Send email to john@test.com with subject 'Hello' and body 'How are you?'", additional_kwargs={}, response_metadata={}, id='d9c49268-8d2c-4cfc-82c5-c4e90db277c5'),
  AIMessage(content='', additional_kwargs={'reasoning_content': "Okay, the user wants to send an email to john@test.com with the subject 'Hello' and body 'How are you?'. Let me check the available tools. There's a send_email_tool that requires recipient, subject, and body. All three are required. The parameters are all strings. The user provided all three: recipient is john@test.com, subject is Hello, body is How are you?. So I need to call send_email_tool with these arguments. No issues here. I'll format the tool_call accordingly.\n", 'tool_calls': [{'id': 'jp9835ge2', 'function': {'arguments': '{"body":"How are you?","recipient":"john@test.com","subject":"Hello"}', 'name': 'send_email_tool'}, 'type': 'function'}]}, response_metadata={'token_usage': {'completion_tokens': 148, 'prompt_token

### Editing

In [33]:
from langchain.agents import create_agent
from langchain.agents.middleware import HumanInTheLoopMiddleware
from langgraph.checkpoint.memory import InMemorySaver


# def read_email_tool(email_id: str) -> str:
#     """Mock function to read an email by its ID."""
#     return f"Email content for ID: {email_id}"

# def send_email_tool(recipient: str, subject: str, body: str) -> str:
#     """Mock function to send an email."""
#     return f"Email sent to {recipient} with subject '{subject}'"

agent = create_agent(
    model="groq:qwen/qwen3-32b",
    tools=[read_email_tool,send_email_tool],
    checkpointer=InMemorySaver(),
    middleware=[
        HumanInTheLoopMiddleware(
            interrupt_on={
                "send_email_tool": {
                    "allowed_decisions": ["approve", "edit", "reject"],
                },
                "read_email_tool": False,
            }
        ),
    ],
)

In [34]:
config = {"configurable": {"thread_id": "test-edit"}}

# Step 1: Request (with wrong info)
result = agent.invoke(
    {"messages": [HumanMessage(content="Send email to wrong@email.com with subject 'Test' and body 'Hello'")]},
    config=config
)

In [35]:
result

{'messages': [HumanMessage(content="Send email to wrong@email.com with subject 'Test' and body 'Hello'", additional_kwargs={}, response_metadata={}, id='c91367fb-0151-478b-8003-bd8ac166b8d8'),
  AIMessage(content='', additional_kwargs={'reasoning_content': "Okay, the user wants to send an email to wrong@email.com with the subject 'Test' and body 'Hello'. Let me check the available tools. There's a send_email_tool that requires recipient, subject, and body. The parameters are all there: recipient is the email address provided, subject is 'Test', and body is 'Hello'. I need to make sure all required fields are included. Yep, all three are required, and the user provided all. So I should call the send_email_tool with those arguments. No need to use the read_email_tool here since the task is about sending, not reading. Just format the tool_call correctly with the parameters.\n", 'tool_calls': [{'id': 'wjnw5ekw5', 'function': {'arguments': '{"body":"Hello","recipient":"wrong@email.com","sub

In [36]:
# Step 2: Edit and approve
if "__interrupt__" in result:
    print("⏸️ Paused! Editing...")
    
    result = agent.invoke(
        Command(
            resume={
                "decisions": [
                    {
                        "type": "edit",
                        "edited_action": {
                            "name": "send_email_tool",      # Tool name
                            "args": {                   # New arguments
                                "recipient": "correct@email.com",
                                "subject": "Corrected Subject",
                                "body": "This was edited by human before sending"
                            }
                        }
                    }
                ]
            }
        ),
        config=config
    )
    
    print(f"✏️ Result: {result['messages'][-1].content}")

⏸️ Paused! Editing...
✏️ Result: Got it! The email has been successfully sent to **correct@email.com** with the subject **"Corrected Subject"** and the body **"This was edited by human before sending"**. Let me know if you need anything else! 📨


In [37]:
result

{'messages': [HumanMessage(content="Send email to wrong@email.com with subject 'Test' and body 'Hello'", additional_kwargs={}, response_metadata={}, id='c91367fb-0151-478b-8003-bd8ac166b8d8'),
  AIMessage(content='', additional_kwargs={'reasoning_content': "Okay, the user wants to send an email to wrong@email.com with the subject 'Test' and body 'Hello'. Let me check the available tools. There's a send_email_tool that requires recipient, subject, and body. The parameters are all there: recipient is the email address provided, subject is 'Test', and body is 'Hello'. I need to make sure all required fields are included. Yep, all three are required, and the user provided all. So I should call the send_email_tool with those arguments. No need to use the read_email_tool here since the task is about sending, not reading. Just format the tool_call correctly with the parameters.\n", 'tool_calls': [{'id': 'wjnw5ekw5', 'function': {'arguments': '{"body":"Hello","recipient":"wrong@email.com","sub